# Annual trend correction — output and rationale

Runs `scripts/build_trend_stats.py` and visualizes what it produces
(`data/count/species_year_statistics.json`) for every modelled species: the fitted
damped-trend level, the multiplier it implies for next year's forecast, and — because a
trend correction is only as good as the evidence that it generalises — the rolling-origin
backtest logic that motivated a *damped* trend over a flat baseline, simple persistence,
or an unconstrained linear extrapolation.

The settled decision and its evidence live in `DECISIONS.md` → "Annual trend
correction"; open follow-ups (rebuild cadence, retrain-gate interaction, alerting on the
clamp) live in `DEVELOPMENT.md`. This notebook is illustration, not the source of truth —
the fitting logic itself is `scripts/build_trend_stats.py::damped_trend_fit` and the
forecast/clamp logic is `src/trend.py::Trend`.

In [ ]:
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils

rootutils.setup_root(".", indicator=".project-root", pythonpath=True)

from scripts.build_trend_stats import TrendBuilder, damped_trend_fit, species_from_experiments
from src.trend import MAX_LOG_MULTIPLIER, PHI, Trend

DATA_DIR = "../data"
CONFIGS_DIR = "../configs"

## Run the script

Regenerates `data/count/species_year_statistics.json` for every species in
`configs/experiment/*.yaml` — a few seconds, no GPU, just aggregating `all_count_processed.csv`.

In [ ]:
result = subprocess.run(
    [sys.executable, "scripts/build_trend_stats.py"],
    cwd="..",
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError(f"build_trend_stats.py exited {result.returncode}")

## Load the output

Through `Trend.load` — the exact path `DefileLitModule.save_predict` uses at forecast time.

In [ ]:
species_list = species_from_experiments(CONFIGS_DIR)
trends = {s: Trend.load(DATA_DIR, s) for s in species_list}
species_list

## Summary table — this year's correction per species

`multiplier_raw` is what the fit alone implies for the next forecast year;
`multiplier_clamped` is what `Trend.multiplier` actually returns once the `[1/3, 3]x`
sanity bound (`MAX_LOG_MULTIPLIER`) is applied. `clamped` flags where the two differ —
currently only Red Kite, whose raw fit (3.41x) sits outside the bound.

In [ ]:
def summarize(species):
    t = trends[species]
    next_year = t.last_year + 1
    log_mult_raw = t.forecast_log_rate(next_year) - t.baseline_log_rate
    return {
        "species": species,
        "n_years": len(t.year),
        "years": f"{t.year[0]}-{t.last_year}",
        "trend_per_year": t.trend,
        "multiplier_raw": float(np.exp(log_mult_raw)),
        "multiplier_clamped": t.multiplier(next_year),
        "clamped": abs(log_mult_raw) > MAX_LOG_MULTIPLIER,
    }


summary = pd.DataFrame([summarize(s) for s in species_list]).sort_values(
    "multiplier_raw", key=lambda s: (s - 1).abs(), ascending=False
)
summary

## Fitted trend and forecast, per species

Solid: observed annual rate (points) and the fitted Holt level (line). Dashed: the
damped-trend forecast extended 15 years past `last_year`, purely to make the *shape* of
the damping visible — the model is only ever actually used one year ahead. Watch how the
dashed line bends and flattens rather than continuing straight: that flattening is
`phi < 1` doing its job. The dotted horizontal lines mark the `[1/3, 3]x` clamp bound
around `baseline_log_rate`; a species whose dashed forecast would cross one is being held
back from what the raw fit implies (see the summary table above).

In [ ]:
species_order = summary["species"].tolist()
horizon = np.arange(1, 16)

fig, axes = plt.subplots(len(species_order), 1, figsize=(9, 2.0 * len(species_order)), sharex=False)
for ax, species in zip(axes, species_order):
    t = trends[species]
    ax.plot(t.year, t.log_rate, "o", ms=3, color="gray", label="observed")
    ax.plot(t.year, t.level_path, "-", color="black", lw=1.3, label="fitted level")

    forecast_years = t.last_year + horizon
    forecast = [t.forecast_log_rate(y) for y in forecast_years]
    ax.plot(forecast_years, forecast, "--", color="tab:red", lw=1.5, label="damped forecast")

    ax.axhline(t.baseline_log_rate, color="tab:blue", lw=0.8, ls=":", label="baseline (multiplier=1)")
    ax.axhline(t.baseline_log_rate + MAX_LOG_MULTIPLIER, color="gray", lw=0.6, ls=":")
    ax.axhline(t.baseline_log_rate - MAX_LOG_MULTIPLIER, color="gray", lw=0.6, ls=":")

    ax.set_title(species, fontsize=9, loc="left")
    ax.set_ylabel("log(birds/hr)")
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout()

## Why a damped trend, not a straight line — the backtest logic

A trend is easy to fit in hindsight, when both past and future are known. The real
question for a forecast is different: fit on years `[0, i)` only, predict year `i`, and
see how wrong that one-step-ahead guess actually was — repeated for every year in the
series ("rolling-origin" / walk-forward validation). No year's forecast is ever allowed to
see its own answer.

Below, four candidate estimators are compared this way on three illustrative species:
**Red Kite** (a real, strong, still-ongoing trend), **Common Buzzard** (no real trend, just
noise), and **Eurasian Kestrel** (a trend that *reversed* around 2011-2012 — the closest
thing in this dataset to "what happens when the future doesn't extend the past").

In [ ]:
def flat_mean(years, y):
    return np.mean(y)


def persistence(years, y, k=3):
    return np.mean(y[-k:])


def ols_trend(years, y):
    A = np.column_stack([np.ones(len(years)), years - years[0]])
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    return beta[0] + beta[1] * (years[-1] - years[0] + 1)


def damped_trend(years, y):
    _, level, trend = damped_trend_fit(y)
    return level + PHI * trend


METHODS = {"flat_mean": flat_mean, "persistence": persistence, "ols_trend": ols_trend, "damped_trend": damped_trend}
MIN_TRAIN_YEARS = 10


def rolling_origin_backtest(species_key, species_name, builder):
    annual = builder.annual_series(species_name)
    years = annual["year"].to_numpy()
    y = np.log(annual["rate"].to_numpy())
    rows = []
    for i in range(MIN_TRAIN_YEARS, len(years)):
        y_past, years_past, actual = y[:i], years[:i], y[i]
        for name, fn in METHODS.items():
            rows.append(
                {"species": species_key, "year": years[i], "method": name, "actual": actual, "pred": fn(years_past, y_past)}
            )
    return pd.DataFrame(rows)


builder = TrendBuilder(data_dir=DATA_DIR, doy=[196, 335])
illustrative = {"red_kite": "Red Kite", "common_buzzard": "Common Buzzard", "kestrel": "Eurasian Kestrel"}
backtest = pd.concat([rolling_origin_backtest(k, v, builder) for k, v in illustrative.items()], ignore_index=True)
backtest["err"] = backtest["actual"] - backtest["pred"]

rmse = backtest.groupby(["species", "method"])["err"].apply(lambda e: np.sqrt(np.mean(e**2))).unstack()
skill = 1 - (rmse.div(rmse["flat_mean"], axis=0)) ** 2
print("RMSE (log seasonal rate):")
display(rmse)
print("\nSkill vs. flat_mean baseline (1 - MSE_method / MSE_flat_mean):")
display(skill)

In [ ]:
fig, axes = plt.subplots(1, len(illustrative), figsize=(14, 4), sharey=False)
for ax, key in zip(axes, illustrative):
    d = backtest[backtest.species == key]
    actual = d[d.method == "flat_mean"][["year", "actual"]]
    ax.plot(actual.year, actual.actual, "o-", color="black", lw=1.3, ms=3, label="actual")
    for m, c in [("flat_mean", "#BBBBBB"), ("persistence", "#4477AA"), ("ols_trend", "#CCBB44"), ("damped_trend", "#EE6677")]:
        dm = d[d.method == m].sort_values("year")
        ax.plot(dm.year, dm.pred, "--", color=c, lw=1.1, alpha=0.9, label=m)
    ax.set_title(key)
    ax.set_ylabel("log(birds/hr)")
axes[0].legend(fontsize=7)
fig.tight_layout()

Read this alongside the skill table above: `damped_trend` and `persistence` track the
actual series closely for Red Kite (a real, sustained trend) without doing obviously worse
than `flat_mean` for Common Buzzard (no real trend) — while `ols_trend`'s straight-line
extrapolation visibly overshoots whenever the recent slope doesn't hold. Kestrel is the
interesting case: watch every method's forecast around 2011-2013, where the series itself
reverses direction.

## The turning-point blind spot, and why the clamp exists

A backtest can only score a method against turning points that actually occurred in the
data. Some species in this dataset have reversed direction; Red Kite, the species leaning
hardest on this correction, has not — at all, in 33 years. That means the strong backtest
skill for Red Kite only demonstrates "rides an ongoing trend well", not "handles a reversal
it has never seen". `MAX_LOG_MULTIPLIER` in `src/trend.py` exists specifically to bound the
damage if/when that trend eventually turns.

In [ ]:
def rolling_sign_changes(log_rate, year, window=7):
    slopes = [np.polyfit(year[i : i + window], log_rate[i : i + window], 1)[0] for i in range(len(year) - window)]
    return int(np.sum(np.diff(np.sign(slopes)) != 0))


reversal_check = pd.DataFrame(
    [
        {
            "species": s,
            "rolling_7yr_sign_changes": rolling_sign_changes(trends[s].log_rate, trends[s].year),
        }
        for s in species_list
    ]
).sort_values("rolling_7yr_sign_changes")
reversal_check

Species with zero sign changes above (Red Kite, at time of writing) have no historical
test case for a reversal at all. For Kestrel specifically, which did reverse around
2011-2012, the rolling-origin backtest cells above show `damped_trend` overshooting right
at the inflection before correcting within about a year once new data confirmed the
reversal — a real, visible cost, smaller than `flat_mean`'s at the same point but not
zero. That is the best empirical evidence available for what a turning point costs; Red
Kite's absence of one means the clamp is the only safeguard against the same thing
happening there, unobserved so far.

## Next steps

- Operational follow-ups (rebuild cadence, retrain-gate interaction, alerting on the
  clamp) are tracked in `DEVELOPMENT.md` → "Annual trend correction — follow-ups", not
  here.
- Re-run this notebook after `scripts/build_trend_stats.py` picks up a new season's data,
  and glance at the summary table — especially whichever species sits closest to the
  clamp — before trusting the refreshed file in production.